In [ ]:
import os
import re
import sys
import json
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    SimpleRewriteRule,
    Rewriter,
    UnifyCodomainRewriter,
)


In [ ]:
print("hello world!")

In [ ]:
# from depccg.parser import EnglishCCGParser

import depccg
from lambeq import ( DepCCGParser,CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
                    CCGBankParseError, CCGBankParser, DepCCGParseError )
from lambeq import RemoveCupsRewriter, UnifyCodomainRewriter, AtomicType, IQPAnsatz
        

parser    = DepCCGParser(model='elmo', device=-1) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

In [ ]:
diagram = parser.sentence2diagram("Alice loves Bob")
diagram = diagram.normal_form().pregroup_normal_form()

diagram = RemoveCupsRewriter()(diagram)
diagram = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)(diagram)

ansatz = IQPAnsatz(
    {AtomicType.SENTENCE: 1, AtomicType.NOUN: 1, AtomicType.PREPOSITIONAL_PHRASE: 0},
    n_layers=2,
    n_single_qubit_params=3,
)

circuit = ansatz(diagram)


In [ ]:
print(type(diagram))
print(type(circuit))
diagram.draw()

In [14]:
_CLEAN_RE = re.compile(r"[^\w\s']")
# parser    = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1,
     AtomicType.PREPOSITIONAL_PHRASE: 0},
    n_layers=2, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    # Rule to delete punctuation boxes (commas, quotes, dashes)
    punc_rule = SimpleRewriteRule(cod=AtomicType.PUNCTUATION, template=Id(AtomicType.PUNCTUATION))

    # remove_pp2 = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.SENTENCE))
    remove_pp = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.NOUN))

    rewriter = Rewriter(
        [
            'coordination', 'determiner',
            'postadverb', 'preadverb',
            'connector', 'auxiliary',
            'prepositional_phrase',
            'subject_rel_pronoun',
            'object_rel_pronoun',
        ]
    )
    rewriter.add_rules(remove_pp, punc_rule, conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

In [20]:
#################################################
#### Mainly to test if GPU works with:       ####
#### sim.run(tk_circ, n_shots=1) and         ####
#### backend.run_circuit(tk_circ, n_shots=1) ####
#################################################


#### ner kartais cia breikalo skaitoma po viena eilute, istraukiami elemntai  ir lygiai taip pat idedami i `data` kintamji??
def load_cnn_extractive(file_path: str, amount: int = None):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if amount and i == amount:
                break
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = record.get("text")
            summary = record.get("summary")
            text_sentences = record.get("text_sentences")
            summary_sentences = record.get("summary_sentences")
            labels = record.get("labels")
            if text is not None or summary is not None:
                data.append(
                    {
                        "text": text,
                        "summary": summary,
                        "text_sentences": text_sentences,
                        "summary_sentences": summary_sentences,
                        "labels": labels,
                    }
                )
            else:
                print(f"{i}. NULL INSTANCE\ntext: {text}\n{summary}\n")
    return data


def load_qe():
    with open("saves/qe.pkl", "rb") as f:
        loaded_data = pickle.load(f)

    dste = loaded_data["dste"]
    length = 0
    for d in dste:
        length += len(d["circuits"])

    print(type(dste[0]["circuits"][0]))
    print(f"Successfully Loaded {length} circuits!")
    return dste


# es = load_qe()

# result = simulator.run(es[0]['circuits'][0]).result()
# print(result.metadata.get("device", "unknown"))


In [ ]:

def testing_lambeq_DepCCG(sentence: str):
    d = parser.sentence2diagram(sentence)
    print(type(d))
    diagram_removed_cups = remove_cups(d)
    print(type(diagram_removed_cups))
    diagram_noraml_form = diagram_removed_cups.normal_form()
    print(type(diagram_noraml_form))
    diagram_pregroup_normal_form = diagram_noraml_form.pregroup_normal_form()
    print(type(diagram_pregroup_normal_form))
    diagram_unified = unify(diagram_pregroup_normal_form)
    print(type(diagram_unified))
    diagram_ansatz = ansatz(diagram_unified)
    print(type(diagram_ansatz))

testing_lambeq_DepCCG("Alice loves Bob")

In [ ]:
# def testing_lambeq_DepCCG(list_of_sentences: List):
#     for sentence in list_of_sentences:

def preprocess_and_encode_DepCCG(dataset):
    encoded_data, errors = [], []
    errs1, errs2, errs3, errs4 = [], [], [], []

    # for i, ds_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
    for i, ds_dict in enumerate(dataset):
        # with open(os.devnull, 'w') as devnull, \
        #  contextlib.redirect_stdout(devnull), \
        #  contextlib.redirect_stderr(devnull):

            text_sentences = ds_dict['text_sentences']
            labels         = ds_dict['labels']

            sentences_simplified = sentence_simplify(text_sentences)
            diagrams, remove     = sent2diagrams(sentences_simplified)
            text_sentences       = remove_by_idx(text_sentences, remove)
            labels               = remove_by_idx(labels, remove)

            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)

            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            # circuits, remove, errs4 = will_train(circuits)
            # text_sentences          = remove_by_idx(text_sentences, remove)
            # labels                  = remove_by_idx(labels, remove)

            encoded_data.append({'circuits': circuits, 'labels': labels, 'original_text_sentences': text_sentences})


            errors += errs1 + errs2 + errs3 + errs4
            print("text_sentences:", text_sentences)
            print("sentences_simplified:", len(sentences_simplified), sentences_simplified)
            print("noramlized_sentences:", len(normalized_diagrams), normalized_diagrams)
            print("circuits:", circuits)
            # print()

    return encoded_data, errors
        

In [ ]:
dataset = load_cnn_extractive(file_path="Dataset/Raw/cnn_dailymail/train.jsonl",amount=10)

In [25]:
encoded_data, errors = preprocess_and_encode_DepCCG(dataset)


Filtering and Encoding dataset:  20%|██        | 2/10 [01:09<04:47, 35.94s/it]2026-02-11 16:56:09,705 - INFO - depccg.allennlp.dataset.supertagging_dataset - Instance with fields:
 	 words: TextField of length 10 with text: 
 		[And, there's, cars, in, the, water, there's, cars, on, fire]
 		and TokenIndexers : {'elmo': 'ELMoTokenCharactersIndexer', 'tokens': 'SingleIdTokenIndexer'} 
 	 metadata: MetadataField (print field.metadata to see specific information). 
 	 weight: TensorField with shape: torch.Size([1]) and dtype: torch.float32. 

Filtering and Encoding dataset:  40%|████      | 4/10 [02:07<03:01, 30.29s/it]2026-02-11 16:57:20,967 - INFO - depccg.allennlp.dataset.supertagging_dataset - Instance with fields:
 	 words: TextField of length 20 with text: 
 		[The, defendant, will, plead, guilty, because, the, defendant, is, in, fact, guilty, of, the,
		charged, offense, the, plea, agreement, said]
 		and TokenIndexers : {'elmo': 'ELMoTokenCharactersIndexer', 'tokens': 'SingleIdTok

In [26]:
with open('Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9.pkl', 'wb') as file:
    pickle.dump(encoded_data, file)

In [35]:
with open('Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

In [36]:
print(loaded_data == encoded_data)

True


In [ ]:
for i, e in enumerate(errors):
    print(f"{i}: {e}")

0: Diagram 0 (cod=n @ n.r @ s @ n.l) does not compose with diagram 1 (dom=s @ s.r @ n.l) ( normalize() )
1: Diagram 0 (cod=n @ n.r @ s @ n.l) does not compose with diagram 1 (dom=s @ s.r @ n.l) ( normalize() )
2: Diagram 0 (cod=n @ n.r @ s @ n.l) does not compose with diagram 1 (dom=s @ s.r @ n.l) ( normalize() )


In [39]:
print(type(dataset[0]['text_sentences']))
print(type(encoded_data[0]["circuits"][0]))

<class 'list'>
<class 'lambeq.backend.quantum.Diagram'>


In [ ]:
for i in range(10):
    print(len(dataset))

In [17]:
#############################################
# Step 2. Preprocessing and lambeq Pipeline
#############################################

def remove_by_idx(ls, remove):
    if not remove:
        return ls
    remove.sort(reverse=True)
    for n in remove:
        ls.pop(n)
    return ls

def sentence_simplify(sentences):
    return [_CLEAN_RE.sub("", s) for s in sentences if s is not None]


def sent2diagrams(sentences):
    diagrams, none_idx = [], []
    for i, sent in enumerate(sentences):
        d = parser.sentence2diagram(sent, tokenised=False, suppress_exceptions=True)
        if d is None:
            none_idx.append(i)
            continue

        diagrams.append(d)
    print(f"Whilst parsing sentences2diagrams, lost {len(sentences) - len(diagrams)} due to Null, out of {len(sentences)}.")

    return diagrams, none_idx


def normalize(sentence_diagrams):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_cups = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if (d is None):
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            d = d.pregroup_normal_form()
            d = unify(d)

        except Exception as e:
            none_idx.append(i)
            errs.append(f"{e} ( normalize() )")
            drop_cups += 1
            continue

        diagrams_normalized.append(d)

    print(f"Dropped {drop_rewrite} diagrams in rewrite, {drop_cups} in cup removal, out of {len(sentence_diagrams)}.")

    return diagrams_normalized, none_idx, errs

def quantum_encode(diagrams: "List"):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ    = ansatz(diagram)
            # circ = circ.to_tk()
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"{e} ( quantum_encode() )")
            remove.append(i)

    return encoded_diagrams, remove, errs

def will_train(
    circuits,
    qubit_limit: int = 40,
    mem_limit_bytes: int = 7 * 2**30,   # 7 GiB
):

    valid, invalid_idxs, errs = [], [], []
    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            needed = 16 * (2 ** tk_circ.n_qubits)
            if needed > mem_limit_bytes:
                raise RuntimeError(f"Needs {needed} bytes > limit {mem_limit_bytes} bytes ({needed/2**20:.0f} MiB > {(mem_limit_bytes/2**20):.0f} MiB). ")

            comp_pass.apply(tk_circ)

            syms = tk_circ.free_symbols()
            if syms:
                bind_map = {s: 0.0 for s in syms}
                tk_circ.symbol_substitution(bind_map)

            simulator.run(tk_circ, n_shots=1)
            # backend.run_circuit(tk_circ, n_shots=1)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"{e} ( will_train() )")
            continue

        valid.append( circ )

    return valid, invalid_idxs, errs



def preprocess_and_encode(dataset):
    encoded_data, errors = [], []
    errs1, errs2, errs3, errs4 = [], [], [], []

    for i, ds_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
        # with open(os.devnull, 'w') as devnull, \
        #  contextlib.redirect_stdout(devnull), \
        #  contextlib.redirect_stderr(devnull):

        text_sentences = ds_dict['text_sentences']
        labels         = ds_dict['labels']

        sentences_simplified = sentence_simplify(text_sentences)
        diagrams, remove     = sent2diagrams(sentences_simplified)
        text_sentences       = remove_by_idx(text_sentences, remove)
        labels               = remove_by_idx(labels, remove)

        normalized_diagrams, remove, errs2 = normalize(diagrams)
        text_sentences                     = remove_by_idx(text_sentences, remove)
        labels                             = remove_by_idx(labels, remove)

        circuits, remove, errs3 = quantum_encode(normalized_diagrams)
        text_sentences          = remove_by_idx(text_sentences, remove)
        labels                  = remove_by_idx(labels, remove)

        circuits, remove, errs4 = will_train(circuits)
        text_sentences          = remove_by_idx(text_sentences, remove)
        labels                  = remove_by_idx(labels, remove)

        encoded_data.append({'circuits': circuits, 'labels': labels, 'original_text_sentences': text_sentences})


        errors += errs1 + errs2 + errs3 + errs4
        print("text_sentences:", text_sentences)
        print("sentences_simplified:", len(sentences_simplified), sentences_simplified)
        print("noramlized_sentences:", len(normalized_diagrams), normalized_diagrams)
        print("circuits:", circuits)
        # print()

    return encoded_data, errors

In [ ]:
def brbrpatapim(filename, left, right):
    dataset    = load_cnn_extractive(filename)
    dst        = dataset[left:right]
    dste, errs = preprocess_and_encode(dst)
    print("\nDone!")
    
    return dste, errs

train, errs_train = brbrpatapim("Dataset/Raw/cnn_dailymail/train.jsonl", 35, 40)